In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# !pip install -q torch_geometric
# !pip install -q class_resolver
# !pip3 install pymatting

In [3]:
import numpy as np
import torch
import random
import copy
import scipy.sparse as sp

from torch.utils.data import TensorDataset, DataLoader, Subset
from torchvision import models
import torch.nn as nn
import torch.nn.functional as nnFn

# torch-geometric imports
from torch_geometric.nn import ARMAConv
from torch_geometric.data import Data
from sklearn.metrics import roc_auc_score, roc_curve, normalized_mutual_info_score


/home/snu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# import os
# import gc
# import numpy as np
# from PIL import Image

# import torch
# from torch.utils.data import TensorDataset, DataLoader

# base_path = "/content/drive/MyDrive/TejaswiAbburi_va797/Dataset/liver_dataset"

# benign_dir = os.path.join(base_path, "Benign")
# malignant_dir = os.path.join(base_path, "Malignant")

# IMG_SIZE = 224

# images = []
# labels = []

# def load_images_from_folder(folder, label):

#     files = []

#     for root, dirs, filenames in os.walk(folder):
#         for file in filenames:

#             if file.lower().endswith((
#                 ".png",
#                 ".jpg",
#                 ".jpeg",
#                 ".bmp",
#                 ".tif",
#                 ".tiff"
#             )):
#                 files.append(os.path.join(root, file))

#     print(f"Loading {len(files)} images from {folder}")

#     for idx, img_path in enumerate(files):

#         try:
#             img = Image.open(img_path).convert("RGB")

#             img = img.resize((IMG_SIZE, IMG_SIZE))

#             img = np.array(img).astype(np.float32) / 255.0

#             img = np.transpose(img, (2, 0, 1))

#             images.append(img)

#             labels.append(label)

#             if idx % 200 == 0:
#                 print(f"Processed {idx} images")

#         except Exception as e:
#             print(f"Skipping {img_path}: {e}")

# load_images_from_folder(benign_dir, 0)

# load_images_from_folder(malignant_dir, 1)

# X = torch.tensor(
#     np.array(images),
#     dtype=torch.float32
# )

# y = torch.tensor(
#     np.array(labels),
#     dtype=torch.long
# )

# print("Images shape:", X.shape)

# print("Labels shape:", y.shape)

# dataset = TensorDataset(X, y)

# final_loader = DataLoader(
#     dataset,
#     batch_size=128,
#     shuffle=True,
#     num_workers=4,
#     pin_memory=True
# )

# device = "cuda" if torch.cuda.is_available() else "cpu"

# print("Using device:", device)

# torch.backends.cudnn.benchmark = True

# vit = torch.hub.load(
#     'facebookresearch/dino:main',
#     'dino_vitb16'
# )

# vit.eval().to(device)

# vit_feats = []

# y_list = []

# with torch.no_grad():

#     for batch_idx, (imgs, lbls) in enumerate(final_loader):

#         imgs = imgs.to(device, non_blocking=True)

#         with torch.amp.autocast(
#             'cuda',
#             dtype=torch.float16
#         ):

#             feats = vit(imgs)

#         vit_feats.append(feats.cpu())

#         y_list.extend(
#             lbls.numpy().tolist()
#         )

#         if batch_idx % 10 == 0:
#             print(
#                 f"Processed batch "
#                 f"{batch_idx}/{len(final_loader)}"
#             )

#         del imgs
#         del feats

# torch.cuda.empty_cache()

# gc.collect()

# F = torch.cat(
#     vit_feats,
#     dim=0
# ).numpy().astype(np.float32)

# y_labels = np.array(
#     y_list
# ).astype(np.int64)

# print("Feature shape:", F.shape)

# print("Label shape:", y_labels.shape)

# features = F

# np.save(
#     "/content/liver_dino_features.npy",
#     features
# )

# np.save(
#     "/content/liver_dino_labels.npy",
#     y_labels
# )

# print("Saved features successfully!")

In [5]:
import numpy as np

feature_path = "/home/snu/Downloads/liver_dino_features.npy"

label_path = "/home/snu/Downloads/liver_dino_labels.npy"

features = np.load(
    feature_path
).astype(np.float32)

y_labels = np.load(
    label_path
).astype(np.int64)

print("Feature shape:", features.shape)

print("Label shape:", y_labels.shape)

print("\nClass distribution:")

unique, counts = np.unique(
    y_labels,
    return_counts=True
)

for cls, cnt in zip(unique, counts):

    print(f"Class {cls}: {cnt}")

Feature shape: (635, 768)
Label shape: (635,)

Class distribution:
Class 0: 200
Class 1: 435


In [6]:
def sim(h1, h2, tau=0.2):
    z1 = nnFn.normalize(h1, dim=-1, p=2)
    z2 = nnFn.normalize(h2, dim=-1, p=2)
    return torch.mm(z1, z2.t()) / tau

def contrastive_loss_wo_cross_network(h1, h2, z):
    f = lambda x: torch.exp(x)
    intra_sim = f(sim(h1, h1))
    inter_sim = f(sim(h1, h2))
    return -torch.log(inter_sim.diag() /
                     (intra_sim.sum(dim=-1) + inter_sim.sum(dim=-1) - intra_sim.diag()))

def contrastive_loss_wo_cross_view(h1, h2, z):
    f = lambda x: torch.exp(x)
    cross_sim = f(sim(h1, z))
    return -torch.log(cross_sim.diag() / cross_sim.sum(dim=-1))

In [7]:
class MLP(nn.Module):
    def __init__(self, inp_size, outp_size, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.PReLU(), # nn.ELU()
            nn.Dropout(0.3),
            nn.Linear(hidden_size, outp_size)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as nnFn
from torch_geometric.nn import ARMAConv

class ARMAEncoder(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, device, activ="ELU",
                 num_stacks=1, num_layers=3):
        super(ARMAEncoder, self).__init__()
        self.device = device

        activations = {
            "SELU": nnFn.selu,
            "SiLU": nnFn.silu,
            "GELU": nnFn.gelu,
            "ELU": nnFn.elu,
            "RELU": nnFn.relu
        }
        self.act = activations.get(activ, nnFn.relu)

        # Layer 1
        self.arma1 = ARMAConv(
            in_channels=input_dim,
            out_channels=hidden_dim,
            num_stacks=num_stacks,
            num_layers=num_layers,
            act=self.act,
            shared_weights=True,
            dropout=0.25
        )
        self.bn1 = nn.BatchNorm1d(hidden_dim)

        # Layer 2
        self.arma2 = ARMAConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            num_stacks=num_stacks,
            num_layers=num_layers,
            act=self.act,
            shared_weights=True,
            dropout=0.25
        )
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        # Layer 3
        self.arma3 = ARMAConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            num_stacks=num_stacks,
            num_layers=num_layers,
            act=self.act,
            shared_weights=True,
            dropout=0.25
        )
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        self.dropout = nn.Dropout(0.3)
        self.mlp = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.arma1(x, edge_index)
        x = self.bn1(x)
        x = self.act(x)
        x = self.dropout(x)

        x = self.arma2(x, edge_index)
        x = self.bn2(x)
        x = self.act(x)
        x = self.dropout(x)

        x = self.arma3(x, edge_index)
        x = self.bn3(x)
        x = self.act(x)
        x = self.dropout(x)

        logits = self.mlp(x)
        return logits


In [9]:
class EMA():  # Moving Average update
    def __init__(self, beta):
        super().__init__()
        self.beta = beta

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1 - self.beta) * new

def update_moving_average(ema_updater, ma_model, current_model):
    for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
        old_weight, up_weight = ma_params.data, current_params.data
        ma_params.data = ema_updater.update_average(old_weight, up_weight)

In [10]:
class ARMA(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_clusters, device, activ, moving_average_decay=0.5, cut=True, collapse_weight=1.0):
        super(ARMA, self).__init__()
        self.device = device
        self.num_clusters = num_clusters
        self.cut = cut
        self.beta = 0.6
        self.collapse_weight = collapse_weight # Added collapse_weight

        self.online_encoder = ARMAEncoder(input_dim, hidden_dim, device, activ)
        self.target_encoder = copy.deepcopy(self.online_encoder)

        activations = {
            "SELU": nnFn.selu,
            "SiLU": nnFn.silu,
            "GELU": nnFn.gelu,
            "RELU": nnFn.relu
        }
        self.act = activations.get(activ, nnFn.elu)
        self.online_predictor = MLP(hidden_dim, num_clusters, hidden_dim)
        self.loss = self.cut_loss if cut else self.modularity_loss
        self.target_ema_updater = EMA(moving_average_decay)

    def reset_moving_average(self):
        del self.target_encoder
        self.target_encoder = None

    def update_ma(self):
        assert self.target_encoder is not None, 'target encoder has not been created yet'
        update_moving_average(self.target_ema_updater, self.target_encoder, self.online_encoder)

    def forward(self, data1, data2):
        x1 = self.online_encoder(data1)
        logits1 = self.online_predictor(x1)
        S1 = nnFn.softmax(logits1, dim=1)

        x2 = self.online_encoder(data2)
        logits2 = self.online_predictor(x2)
        S2 = nnFn.softmax(logits2, dim=1)

        with torch.no_grad():
            target_proj_one = self.target_encoder(data1).detach()
            target_proj_two = self.target_encoder(data2).detach()

        l1 = self.beta * contrastive_loss_wo_cross_network(x1, x2, target_proj_two) + \
             (1.0 - self.beta) * contrastive_loss_wo_cross_view(x1, x2, target_proj_two)

        l2 = self.beta * contrastive_loss_wo_cross_network(x2, x1, target_proj_one) + \
             (1.0 - self.beta) * contrastive_loss_wo_cross_view(x2, x1, target_proj_one)

        return S1, S2, logits1, logits2, l1, l2

    def modularity_loss(self, A, S):
        C = nnFn.softmax(S, dim=1)
        d = torch.sum(A, dim=1)
        m = torch.sum(A)
        B = A - torch.ger(d, d) / (2 * m)

        I_S = torch.eye(self.num_clusters, device=self.device)
        k = torch.tensor(self.num_clusters, dtype=torch.float32, device=self.device)
        n = S.shape[0]

        modularity_term = (-1 / (2 * m)) * torch.trace(torch.mm(torch.mm(C.t(), B), C))
        collapse_reg_term = (torch.sqrt(k) / n) * torch.norm(torch.sum(C, dim=0), p='fro') - 1

        # Use self.collapse_weight here
        return modularity_term + self.collapse_weight * collapse_reg_term

    def cut_loss(self, A, S):
        S = nnFn.softmax(S, dim=1)
        A_pool = torch.matmul(torch.matmul(A, S).t(), S)
        num = torch.trace(A_pool)

        D = torch.diag(torch.sum(A, dim=-1))
        D_pooled = torch.matmul(torch.matmul(D, S).t(), S)
        den = torch.trace(D_pooled)
        mincut_loss = -(num / den)

        St_S = torch.matmul(S.t(), S)
        I_S = torch.eye(self.num_clusters, device=self.device)
        ortho_loss = torch.norm(St_S / torch.norm(St_S) - I_S / torch.norm(I_S))

        return mincut_loss + ortho_loss

In [11]:
def create_adj(features, cut, alpha=1.0):
    """Return a dense W0 matrix (only once), as you originally used for A1 / unsup loss.
       We still create the dense matrix once, but all augmentations below work with edge_index.
    """
    F_norm = features / np.linalg.norm(features, axis=1, keepdims=True)
    W = np.dot(F_norm, F_norm.T)

    if cut == 0:
        W = np.where(W >= alpha, 1, 0).astype(np.float32)
        W = (W / W.max()).astype(np.float32)
    else:
        W = (W * (W >= alpha)).astype(np.float32)
    return W

In [12]:
def edge_index_from_dense(W):
    """Return edge_index as numpy array shape (2, E) and edge_weight vector."""
    rows, cols = np.nonzero(W > 0)
    edge_index = np.vstack([rows, cols]).astype(np.int64)
    edge_weight = W[rows, cols].astype(np.float32)
    return edge_index, edge_weight

In [13]:
def build_adj_list(edge_index_np, num_nodes):
    """Build adjacency list: list of neighbor arrays for each node (numpy)."""
    adj = [[] for _ in range(num_nodes)]
    src = edge_index_np[0]
    dst = edge_index_np[1]
    for s, d in zip(src, dst):
        adj[s].append(d)
    # convert to numpy arrays for speed
    adj = [np.array(neis, dtype=np.int64) if len(neis) > 0 else np.array([], dtype=np.int64) for neis in adj]
    return adj

In [14]:
def aug_random_edge_edge_index(edge_index_np, drop_percent=0.2, seed=None):
    """Randomly drop edges from edge_index. Returns new edge_index (2 x E') and edge_weight placeholder."""
    rng = np.random.default_rng(seed)
    E = edge_index_np.shape[1]
    keep_mask = rng.random(E) >= drop_percent
    new_edge_index = edge_index_np[:, keep_mask]
    return new_edge_index

In [15]:
def aug_subgraph_edge_index(features_np, edge_index_np, adj_list, drop_percent=0.2, seed=None):
    """
    Sample a subgraph by selecting s_node_num nodes via neighbor expansion (BFS-like),
    then return (sub_features, sub_edge_index) with node ids remapped to [0..s-1].
    """
    rng = np.random.default_rng(seed)
    num_nodes = features_np.shape[0]
    s_node_num = int(num_nodes * (1 - drop_percent))
    if s_node_num < 1:
        s_node_num = 1

    # choose a random center node
    center_node = int(rng.integers(0, num_nodes))
    sub_nodes = [center_node]
    front_idx = 0

    # BFS-like expansion using adjacency list until we reach s_node_num
    while len(sub_nodes) < s_node_num and front_idx < len(sub_nodes):
        cur = sub_nodes[front_idx]
        neighbors = adj_list[cur]
        if neighbors.size > 0:
            # shuffle neighbors and try to add new ones
            nbrs_shuffled = neighbors.copy()
            rng.shuffle(nbrs_shuffled)
            for nb in nbrs_shuffled:
                if nb not in sub_nodes:
                    sub_nodes.append(int(nb))
                    if len(sub_nodes) >= s_node_num:
                        break
        front_idx += 1
        # if BFS stalls (no new neighbors), add random nodes
        if front_idx >= len(sub_nodes) and len(sub_nodes) < s_node_num:
            remaining = [n for n in range(num_nodes) if n not in sub_nodes]
            if not remaining:
                break
            add = int(rng.choice(remaining))
            sub_nodes.append(add)

    sub_nodes = sorted(set(sub_nodes))
    node_map = {old: new for new, old in enumerate(sub_nodes)}

    # induce edges that have both ends in sub_nodes
    src = edge_index_np[0]
    dst = edge_index_np[1]
    mask_src_in = np.isin(src, sub_nodes)
    mask_dst_in = np.isin(dst, sub_nodes)
    mask = mask_src_in & mask_dst_in
    sel_src = src[mask]
    sel_dst = dst[mask]
    # remap
    remapped_src = np.array([node_map[int(s)] for s in sel_src], dtype=np.int64)
    remapped_dst = np.array([node_map[int(d)] for d in sel_dst], dtype=np.int64)
    new_edge_index = np.vstack([remapped_src, remapped_dst])
    # sub features
    sub_features = features_np[sub_nodes, :].astype(np.float32)
    return sub_features, new_edge_index

In [16]:
def load_data_from_edge_index(node_feats_np, edge_index_np, device):
    """Return PyG Data with torch tensors. edge_index_np is (2, E) numpy."""
    node_feats = torch.from_numpy(node_feats_np).float()
    edge_index = torch.from_numpy(edge_index_np.astype(np.int64)).long()
    return node_feats, edge_index

In [17]:
cut = 1 # Consider n-cut loss OR Modularity loss (by default cut = 0)
alpha = 0.83 # Edge creation Threshold
device = 'cuda' if torch.cuda.is_available() else 'cpu'
K = 2  # Number of clusters
np.random.seed(42)
# Define all activation functions to test
define_activations = ["SELU", "SiLU", "GELU", "ELU", "RELU"]
activ = "SiLU"
num_epochs = 5000
base_seed = 42
lambda_contrastive = 0.010 # Updated based on performance analysis
feats_dim = features.shape[1]

In [18]:
# cut = 0 # Consider n-cut loss OR Modularity loss (by default cut = 0)
# alpha = 0.83 # Edge creation Threshold
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# K = 2  # Number of clusters
# np.random.seed(42)
# # Define all activation functions to test
# define_activations = ["SELU", "SiLU", "GELU", "ELU", "RELU"]
# activ = "SiLU"
# num_epochs = 5000
# base_seed = 42
# lambda_contrastive = 0.010 # Updated based on performance analysis
# feats_dim = features.shape[1]

In [19]:
W0 = create_adj(features, cut, alpha)  # shape (N, N) dense
A1 = torch.from_numpy(W0).float().to(device)

edge_index_np, edge_weight_np = edge_index_from_dense(W0)  # numpy edge_index (2, E)
num_nodes = features.shape[0]
adj_list = build_adj_list(edge_index_np, num_nodes)  # adjacency list for fast subgraph sampling

# convert features to numpy (we'll slice them in augmentations)
features_np = features.copy()

# Build initial Data object (full graph)
node_feats_full, edge_index_full = load_data_from_edge_index(features_np, edge_index_np, device)
data0 = Data(x=node_feats_full.to(device), edge_index=edge_index_full.to(device))
print("Data0:", data0)

Data0: Data(x=[635, 768], edge_index=[2, 65773])


In [20]:
from torch.optim.lr_scheduler import StepLR
from torch.optim import AdamW

feats_dim = features.shape[1]
model = ARMA(feats_dim, 256, K, device, activ, cut).to(device) # Increased hidden_dim from 64 to 256
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=200, gamma=0.5)

# Seeds
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

# Training hyperparams (you can reduce num_epochs for debugging)
num_epochs = 2500 # Increased number of epochs
lambda_contrastive = 0.7 # Increased weight for contrastive loss
for epoch in range(num_epochs):
    # --- Augmentations using edge_index or adjacency list (fast, sparse) ---
    # 1) Random edge drop on edge_index
    W_aug1_edge_index = aug_random_edge_edge_index(edge_index_np, drop_percent=0.2, seed=epoch)

    # 2) Subgraph via adjacency list (returns sub_features and sub_edge_index)
    W_aug2_edge_index = aug_random_edge_edge_index(edge_index_np, drop_percent=0.2, seed=epoch + 999)
    features_aug2 = features_np.copy()

    # 3) Feature augmentations (keep these as numpy operations)
    # Feature dropout (column-wise)
    rng = np.random.default_rng(epoch)
    mask = rng.random(features_np.shape) >= 0.2
    features_aug1 = (features_np * mask.astype(np.float32))

    # Feature cell dropout (random cell zeroing)
    aug_feat2 = features_np.copy()
    num_nodes_local, feat_dim = aug_feat2.shape
    drop_feat_num = int(num_nodes_local * feat_dim * 0.2)
    # random positions to zero
    flat_idx = rng.choice(num_nodes_local * feat_dim, size=drop_feat_num, replace=False)
    rows = (flat_idx // feat_dim)
    cols = (flat_idx % feat_dim)
    aug_feat2[rows, cols] = 0.0
    features_aug2_feat = aug_feat2.astype(np.float32)

    # --- Build PyG Data objects for the two views ---
    # view1: features_aug1 with W_aug1_edge_index
    node_feats1, edge_index1 = load_data_from_edge_index(features_aug1, W_aug1_edge_index, device)
    data1 = Data(x=node_feats1.to(device), edge_index=edge_index1.to(device))

    # view2: features_aug2 (from subgraph) and its edge_index
    node_feats2, edge_index2 = load_data_from_edge_index(features_aug2, W_aug2_edge_index, device)
    data2 = Data(x=node_feats2.to(device), edge_index=edge_index2.to(device))

    # --- Training step ---
    model.train()
    optimizer.zero_grad()

    S1, S2, logits1, logits2, l1, l2 = model(data1, data2)

    unsup_loss = model.loss(A1, logits1)
    cont_loss = ((l1 + l2) / 2).mean()
    total_loss = unsup_loss + lambda_contrastive * cont_loss

    total_loss.backward()
    optimizer.step()
    scheduler.step()
    model.update_ma()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Total: {total_loss.item():.4f} | Unsup: {unsup_loss.item():.4f} | Cont: {cont_loss.item():.4f}")

Epoch 0 | Total: 4.5248 | Unsup: -0.2498 | Cont: 6.8209
Epoch 100 | Total: 3.3305 | Unsup: -0.7466 | Cont: 5.8244
Epoch 200 | Total: 3.0791 | Unsup: -0.8027 | Cont: 5.5455
Epoch 300 | Total: 3.0344 | Unsup: -0.8123 | Cont: 5.4953
Epoch 400 | Total: 2.9686 | Unsup: -0.8138 | Cont: 5.4034
Epoch 500 | Total: 2.9426 | Unsup: -0.8179 | Cont: 5.3721
Epoch 600 | Total: 2.9244 | Unsup: -0.8208 | Cont: 5.3503
Epoch 700 | Total: 2.9000 | Unsup: -0.8249 | Cont: 5.3212
Epoch 800 | Total: 2.9038 | Unsup: -0.8235 | Cont: 5.3248
Epoch 900 | Total: 2.8861 | Unsup: -0.8247 | Cont: 5.3011
Epoch 1000 | Total: 2.8830 | Unsup: -0.8192 | Cont: 5.2888
Epoch 1100 | Total: 2.8900 | Unsup: -0.8244 | Cont: 5.3062
Epoch 1200 | Total: 2.8745 | Unsup: -0.8257 | Cont: 5.2860
Epoch 1300 | Total: 2.8693 | Unsup: -0.8267 | Cont: 5.2800
Epoch 1400 | Total: 2.8682 | Unsup: -0.8266 | Cont: 5.2783
Epoch 1500 | Total: 2.8799 | Unsup: -0.8225 | Cont: 5.2892
Epoch 1600 | Total: 2.8747 | Unsup: -0.8230 | Cont: 5.2824
Epoch 170

In [21]:
model.eval()
with torch.no_grad():
    S1, _, logits1, _, _, _ = model(data0, data0)
    y_pred_proba = nnFn.softmax(logits1, dim=1).cpu().numpy()
    y_pred = np.argmax(y_pred_proba, axis=1)

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, normalized_mutual_info_score
acc_score = accuracy_score(y_labels, y_pred)
acc_score_inverted = accuracy_score(y_labels, 1 - y_pred)
if acc_score_inverted > acc_score:
    acc_score = acc_score_inverted
    y_pred = 1 - y_pred

prec_score = precision_score(y_labels, y_pred)
rec_score = recall_score(y_labels, y_pred)
f1 = f1_score(y_labels, y_pred)
log_loss_value = log_loss(y_labels, y_pred_proba)
nmi = normalized_mutual_info_score(y_labels, y_pred, average_method='arithmetic')

print("Accuracy Score:", acc_score)
print("Precision Score:", prec_score)
print("Recall Score:", rec_score)
print("F1 Score:", f1)
print("Log Loss:", log_loss_value)
print("NMI:", nmi)

Accuracy Score: 0.6897637795275591
Precision Score: 0.8269230769230769
Recall Score: 0.6919540229885057
F1 Score: 0.753441802252816
Log Loss: 8.430797964750841
NMI: 0.09670402518678081


In [23]:
from collections import Counter

print("\n===== CLUSTER DISTRIBUTION =====")
print(Counter(y_pred))


===== CLUSTER DISTRIBUTION =====
Counter({1: 364, 0: 271})


In [24]:
print(y_pred[:20])
print(y_pred_proba.max(axis=-1)[:20])

[1 0 1 0 0 1 1 0 0 0 1 1 0 1 1 0 0 1 0 0]
[0.6843736  1.         1.         0.9999995  1.         0.99999964
 0.99291444 0.99998105 1.         1.         0.99998534 0.99993145
 0.99999976 0.9998323  0.99999917 1.         0.99999964 0.9999528
 0.9995197  0.9938069 ]


In [25]:
num_runs = 10
num_epochs = 5000
lr = 1e-4
weight_decay = 1e-4
lambda_list = [0.001, 0.005, 0.009, 0.01, 0.05, 0.09, 0.1, 0.3, 0.5, 0.9, 1, 2, 5, 8]
base_seed = 42
cut = 1
all_results = []

for lam in lambda_list:
    print(f"\n================ LAMBDA = {lam} ================\n")

    acc_scores, prec_scores, rec_scores, f1_scores, log_losses = [], [], [], [], []

    for run in range(num_runs):
        print(f"\n--- Run {run + 1}/{num_runs} ---")
        torch.manual_seed(base_seed + run)
        np.random.seed(base_seed + run)
        random.seed(base_seed + run)

        # --- Model Setup ---
        model = ARMA(feats_dim, 256, K, device, activ, cut).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = StepLR(optimizer, step_size=200, gamma=0.5)

        for epoch in range(num_epochs):
            # 1) Two random edge augmentations
            W_aug1_edge_index = aug_random_edge_edge_index(edge_index_np, drop_percent=0.2, seed=epoch)
            W_aug2_edge_index = aug_random_edge_edge_index(edge_index_np, drop_percent=0.2, seed=epoch + 999)

            # 2) Feature augmentations
            rng = np.random.default_rng(epoch)
            mask = rng.random(features_np.shape) >= 0.2
            features_aug1 = (features_np * mask.astype(np.float32))
            features_aug2 = features_np.copy()
            num_nodes_local, feat_dim = features_aug2.shape
            drop_feat_num = int(num_nodes_local * feat_dim * 0.2)
            flat_idx = rng.choice(num_nodes_local * feat_dim, size=drop_feat_num, replace=False)
            rows = (flat_idx // feat_dim)
            cols = (flat_idx % feat_dim)
            features_aug2[rows, cols] = 0.0
            features_aug2_feat = features_aug2.astype(np.float32)

            # 3) Build Data views
            node_feats1, edge_index1 = load_data_from_edge_index(features_aug1, W_aug1_edge_index, device)
            data1 = Data(x=node_feats1.to(device), edge_index=edge_index1.to(device))
            node_feats2, edge_index2 = load_data_from_edge_index(features_aug2_feat, W_aug2_edge_index, device)
            data2 = Data(x=node_feats2.to(device), edge_index=edge_index2.to(device))

            # --- Training step ---
            model.train()
            optimizer.zero_grad()

            S1, S2, logits1, logits2, l1, l2 = model(data1, data2)
            unsup_loss = model.loss(A1, logits1)
            cont_loss = ((l1 + l2) / 2).mean()
            total_loss = unsup_loss + lam * cont_loss

            total_loss.backward()
            optimizer.step()
            scheduler.step()
            model.update_ma()

            if epoch % 500 == 0:
                print(f"Epoch {epoch} | Total: {total_loss.item():.4f} | Unsup: {unsup_loss.item():.4f} | Cont: {cont_loss.item():.4f}")

        # --- Evaluation ---
        model.eval()
        with torch.no_grad():
            S1, _, logits1, _, _, _ = model(data0, data0)
            y_pred_proba = nnFn.softmax(logits1, dim=1).cpu().numpy()
            y_pred = np.argmax(y_pred_proba, axis=1)

        acc = accuracy_score(y_labels, y_pred)
        acc_inv = accuracy_score(y_labels, 1 - y_pred)
        if acc_inv > acc:
            acc = acc_inv
            y_pred = 1 - y_pred

        prec = precision_score(y_labels, y_pred)
        rec = recall_score(y_labels, y_pred)
        f1 = f1_score(y_labels, y_pred)
        ll = log_loss(y_labels, y_pred_proba)

        acc_scores.append(acc)
        prec_scores.append(prec)
        rec_scores.append(rec)
        f1_scores.append(f1)
        log_losses.append(ll)

        print(f"Run {run + 1} Accuracy: {acc:.4f}, F1: {f1:.4f}")

    # --- Aggregate Results ---
    lambda_results = {
        "lambda": lam,
        "accuracy": (np.mean(acc_scores), np.std(acc_scores)),
        "precision": (np.mean(prec_scores), np.std(prec_scores)),
        "recall": (np.mean(rec_scores), np.std(rec_scores)),
        "f1": (np.mean(f1_scores), np.std(f1_scores)),
        "log_loss": (np.mean(log_losses), np.std(log_losses))
    }
    all_results.append(lambda_results)

    print(f"\n--- RESULTS FOR LAMBDA = {lam} ---")
    print(f"Accuracy: {lambda_results['accuracy'][0]:.4f} ± {lambda_results['accuracy'][1]:.4f}")
    print(f"Precision: {lambda_results['precision'][0]:.4f} ± {lambda_results['precision'][1]:.4f}")
    print(f"Recall: {lambda_results['recall'][0]:.4f} ± {lambda_results['recall'][1]:.4f}")
    print(f"F1 Score: {lambda_results['f1'][0]:.4f} ± {lambda_results['f1'][1]:.4f}")
    print(f"Log Loss: {lambda_results['log_loss'][0]:.4f} ± {lambda_results['log_loss'][1]:.4f}")

# ==========================================================
# === Final Summary ===
# ==========================================================
print("\n================ FINAL SUMMARY FOR ALL LAMBDAS ================\n")
print(f"{'Lambda':>8} | {'Accuracy':>18} | {'Precision':>18} | {'Recall':>18} | {'F1 Score':>18} | {'Log Loss':>18}")
print("-" * 108)
for res in all_results:
    print(f"{res['lambda']:>8} | "
          f"{res['accuracy'][0]:.4f} ± {res['accuracy'][1]:.4f} | "
          f"{res['precision'][0]:.4f} ± {res['precision'][1]:.4f} | "
          f"{res['recall'][0]:.4f} ± {res['recall'][1]:.4f} | "
          f"{res['f1'][0]:.4f} ± {res['f1'][1]:.4f} | "
          f"{res['log_loss'][0]:.4f} ± {res['log_loss'][1]:.4f}")


================ LAMBDA = 0.001 ================


--- Run 1/10 ---
Epoch 0 | Total: -0.2419 | Unsup: -0.2487 | Cont: 6.8475



KeyboardInterrupt



In [26]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, log_loss, normalized_mutual_info_score
)
import numpy as np
import random
import torch

num_runs = 10
num_epochs = 2500
lambda_contrastive = 8
cut = 1
acc_list, prec_list, rec_list, f1_list, logloss_list, nmi_list = [], [], [], [], [], []

for run in range(num_runs):
    print(f"\n===== RUN {run+1} =====")

    seed = 42 + run
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model     = ARMA(feats_dim, 256, K, device, activ, cut).to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = StepLR(optimizer, step_size=200, gamma=0.5)

    # -------------------- Training --------------------
    for epoch in range(num_epochs):

        W_aug1_edge_index = aug_random_edge_edge_index(
            edge_index_np, drop_percent=0.2, seed=epoch + seed)
        W_aug2_edge_index = aug_random_edge_edge_index(
            edge_index_np, drop_percent=0.2, seed=epoch + 999 + seed)

        rng  = np.random.default_rng(epoch + seed)
        mask = rng.random(features_np.shape) >= 0.2
        features_aug1 = features_np * mask.astype(np.float32)

        aug_feat2 = features_np.copy()
        num_nodes_local, feat_dim = aug_feat2.shape
        drop_feat_num = int(num_nodes_local * feat_dim * 0.2)
        flat_idx = rng.choice(num_nodes_local * feat_dim, size=drop_feat_num, replace=False)
        rows = flat_idx // feat_dim
        cols = flat_idx % feat_dim
        aug_feat2[rows, cols] = 0.0
        features_aug2 = aug_feat2.astype(np.float32)

        node_feats1, edge_index1 = load_data_from_edge_index(
            features_aug1, W_aug1_edge_index, device)
        data1 = Data(x=node_feats1.to(device), edge_index=edge_index1.to(device))

        node_feats2, edge_index2 = load_data_from_edge_index(
            features_aug2, W_aug2_edge_index, device)
        data2 = Data(x=node_feats2.to(device), edge_index=edge_index2.to(device))

        model.train()
        optimizer.zero_grad()

        S1, S2, logits1, logits2, l1, l2 = model(data1, data2)

        unsup_loss = model.loss(A1, logits1)
        cont_loss  = ((l1 + l2) / 2).mean()
        total_loss = unsup_loss + lambda_contrastive * cont_loss

        total_loss.backward()
        optimizer.step()
        scheduler.step()
        model.update_ma()

    model.eval()
    with torch.no_grad():
        S1, _, logits1, _, _, _ = model(data0, data0)
        y_pred_proba = torch.nn.functional.softmax(logits1, dim=1).cpu().numpy()
        y_pred       = np.argmax(y_pred_proba, axis=1)

    acc_score     = accuracy_score(y_labels, y_pred)
    acc_score_inv = accuracy_score(y_labels, 1 - y_pred)
    if acc_score_inv > acc_score:
        y_pred    = 1 - y_pred
        acc_score = acc_score_inv

    prec    = precision_score(y_labels, y_pred, zero_division=0)
    rec     = recall_score(y_labels, y_pred,    zero_division=0)
    f1_val  = f1_score(y_labels, y_pred,        zero_division=0)
    logloss = log_loss(y_labels, y_pred_proba)


    nmi = normalized_mutual_info_score(y_labels, y_pred, average_method='arithmetic')

    print(f"Acc: {acc_score:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | "
          f"F1: {f1_val:.4f} | NMI: {nmi:.4f}")

    acc_list.append(acc_score)
    prec_list.append(prec)
    rec_list.append(rec)
    f1_list.append(f1_val)
    logloss_list.append(logloss)
    nmi_list.append(nmi)

# -------------------- Final Results --------------------
print("\n===== FINAL RESULTS (10 RUNS) =====")
print(f"Accuracy : {np.mean(acc_list):.4f}  ± {np.std(acc_list):.4f}")
print(f"Precision: {np.mean(prec_list):.4f} ± {np.std(prec_list):.4f}")
print(f"Recall   : {np.mean(rec_list):.4f}  ± {np.std(rec_list):.4f}")
print(f"F1-score : {np.mean(f1_list):.4f}  ± {np.std(f1_list):.4f}")
print(f"NMI      : {np.mean(nmi_list):.4f}  ± {np.std(nmi_list):.4f}")
print(f"Log Loss : {np.mean(logloss_list):.4f}  ± {np.std(logloss_list):.4f}")


===== RUN 1 =====
Acc: 0.7386 | Prec: 0.8180 | Rec: 0.7954 | F1: 0.8065 | NMI: 0.1268

===== RUN 2 =====
Acc: 0.7150 | Prec: 0.8053 | Rec: 0.7701 | F1: 0.7873 | NMI: 0.0983

===== RUN 3 =====
Acc: 0.7181 | Prec: 0.8699 | Rec: 0.6920 | F1: 0.7708 | NMI: 0.1496

===== RUN 4 =====
Acc: 0.7307 | Prec: 0.8300 | Rec: 0.7632 | F1: 0.7952 | NMI: 0.1282

===== RUN 5 =====
Acc: 0.7339 | Prec: 0.8244 | Rec: 0.7770 | F1: 0.8000 | NMI: 0.1269

===== RUN 6 =====
Acc: 0.6976 | Prec: 0.7416 | Rec: 0.8575 | F1: 0.7953 | NMI: 0.0470

===== RUN 7 =====
Acc: 0.7433 | Prec: 0.8148 | Rec: 0.8092 | F1: 0.8120 | NMI: 0.1293

===== RUN 8 =====
Acc: 0.7433 | Prec: 0.8269 | Rec: 0.7908 | F1: 0.8085 | NMI: 0.1374

===== RUN 9 =====
Acc: 0.7339 | Prec: 0.8595 | Rec: 0.7310 | F1: 0.7901 | NMI: 0.1537

===== RUN 10 =====
Acc: 0.7244 | Prec: 0.8283 | Rec: 0.7540 | F1: 0.7894 | NMI: 0.1217

===== FINAL RESULTS (10 RUNS) =====
Accuracy : 0.7279  ± 0.0136
Precision: 0.8219 ± 0.0326
Recall   : 0.7740  ± 0.0425
F1-score 

In [ ]:
# num_runs = 10
# num_epochs = 5000
# lr = 1e-4
# weight_decay = 1e-4
# lambda_list = [0.001, 0.005, 0.009, 0.01, 0.05, 0.09, 0.1, 0.3, 0.5, 0.9, 1, 2, 5, 8]
# base_seed = 42

# all_results = []

# for lam in lambda_list:
#     print(f"\n================ LAMBDA = {lam} ================\n")

#     acc_scores, prec_scores, rec_scores, f1_scores, log_losses = [], [], [], [], []

#     for run in range(num_runs):
#         print(f"\n--- Run {run + 1}/{num_runs} ---")
#         torch.manual_seed(base_seed + run)
#         np.random.seed(base_seed + run)
#         random.seed(base_seed + run)

#         # --- Model Setup ---
#         model = ARMA(feats_dim, 256, K, device, activ, cut).to(device)
#         optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
#         scheduler = StepLR(optimizer, step_size=200, gamma=0.5)

#         for epoch in range(num_epochs):
#             # 1) Two random edge augmentations
#             W_aug1_edge_index = aug_random_edge_edge_index(edge_index_np, drop_percent=0.2, seed=epoch)
#             W_aug2_edge_index = aug_random_edge_edge_index(edge_index_np, drop_percent=0.2, seed=epoch + 999)

#             # 2) Feature augmentations
#             rng = np.random.default_rng(epoch)
#             mask = rng.random(features_np.shape) >= 0.2
#             features_aug1 = (features_np * mask.astype(np.float32))
#             features_aug2 = features_np.copy()
#             num_nodes_local, feat_dim = features_aug2.shape
#             drop_feat_num = int(num_nodes_local * feat_dim * 0.2)
#             flat_idx = rng.choice(num_nodes_local * feat_dim, size=drop_feat_num, replace=False)
#             rows = (flat_idx // feat_dim)
#             cols = (flat_idx % feat_dim)
#             features_aug2[rows, cols] = 0.0
#             features_aug2_feat = features_aug2.astype(np.float32)

#             # 3) Build Data views
#             node_feats1, edge_index1 = load_data_from_edge_index(features_aug1, W_aug1_edge_index, device)
#             data1 = Data(x=node_feats1.to(device), edge_index=edge_index1.to(device))
#             node_feats2, edge_index2 = load_data_from_edge_index(features_aug2_feat, W_aug2_edge_index, device)
#             data2 = Data(x=node_feats2.to(device), edge_index=edge_index2.to(device))

#             # --- Training step ---
#             model.train()
#             optimizer.zero_grad()

#             S1, S2, logits1, logits2, l1, l2 = model(data1, data2)
#             unsup_loss = model.loss(A1, logits1)
#             cont_loss = ((l1 + l2) / 2).mean()
#             total_loss = unsup_loss + lam * cont_loss

#             total_loss.backward()
#             optimizer.step()
#             scheduler.step()
#             model.update_ma()

#             if epoch % 500 == 0:
#                 print(f"Epoch {epoch} | Total: {total_loss.item():.4f} | Unsup: {unsup_loss.item():.4f} | Cont: {cont_loss.item():.4f}")

#         # --- Evaluation ---
#         model.eval()
#         with torch.no_grad():
#             S1, _, logits1, _, _, _ = model(data0, data0)
#             y_pred_proba = nnFn.softmax(logits1, dim=1).cpu().numpy()
#             y_pred = np.argmax(y_pred_proba, axis=1)

#         acc = accuracy_score(y_labels, y_pred)
#         acc_inv = accuracy_score(y_labels, 1 - y_pred)
#         if acc_inv > acc:
#             acc = acc_inv
#             y_pred = 1 - y_pred

#         prec = precision_score(y_labels, y_pred)
#         rec = recall_score(y_labels, y_pred)
#         f1 = f1_score(y_labels, y_pred)
#         ll = log_loss(y_labels, y_pred_proba)

#         acc_scores.append(acc)
#         prec_scores.append(prec)
#         rec_scores.append(rec)
#         f1_scores.append(f1)
#         log_losses.append(ll)

#         print(f"Run {run + 1} Accuracy: {acc:.4f}, F1: {f1:.4f}")

#     # --- Aggregate Results ---
#     lambda_results = {
#         "lambda": lam,
#         "accuracy": (np.mean(acc_scores), np.std(acc_scores)),
#         "precision": (np.mean(prec_scores), np.std(prec_scores)),
#         "recall": (np.mean(rec_scores), np.std(rec_scores)),
#         "f1": (np.mean(f1_scores), np.std(f1_scores)),
#         "log_loss": (np.mean(log_losses), np.std(log_losses))
#     }
#     all_results.append(lambda_results)

#     print(f"\n--- RESULTS FOR LAMBDA = {lam} ---")
#     print(f"Accuracy: {lambda_results['accuracy'][0]:.4f} ± {lambda_results['accuracy'][1]:.4f}")
#     print(f"Precision: {lambda_results['precision'][0]:.4f} ± {lambda_results['precision'][1]:.4f}")
#     print(f"Recall: {lambda_results['recall'][0]:.4f} ± {lambda_results['recall'][1]:.4f}")
#     print(f"F1 Score: {lambda_results['f1'][0]:.4f} ± {lambda_results['f1'][1]:.4f}")
#     print(f"Log Loss: {lambda_results['log_loss'][0]:.4f} ± {lambda_results['log_loss'][1]:.4f}")

# # ==========================================================
# # === Final Summary ===
# # ==========================================================
# print("\n================ FINAL SUMMARY FOR ALL LAMBDAS ================\n")
# print(f"{'Lambda':>8} | {'Accuracy':>18} | {'Precision':>18} | {'Recall':>18} | {'F1 Score':>18} | {'Log Loss':>18}")
# print("-" * 108)
# for res in all_results:
#     print(f"{res['lambda']:>8} | "
#           f"{res['accuracy'][0]:.4f} ± {res['accuracy'][1]:.4f} | "
#           f"{res['precision'][0]:.4f} ± {res['precision'][1]:.4f} | "
#           f"{res['recall'][0]:.4f} ± {res['recall'][1]:.4f} | "
#           f"{res['f1'][0]:.4f} ± {res['f1'][1]:.4f} | "
#           f"{res['log_loss'][0]:.4f} ± {res['log_loss'][1]:.4f}")

=============== FINAL SUMMARY FOR ALL LAMBDAS ================

  Lambda |           Accuracy |          Precision |             Recall |           F1 Score |           Log Loss
------------------------------------------------------------------------------------------------------------
   0.001 | 0.7276 ± 0.0055 | 0.8048 ± 0.0046 | 0.7952 ± 0.0073 | 0.7999 ± 0.0044 | 3.9280 ± 2.8151
   0.005 | 0.7282 ± 0.0058 | 0.8064 ± 0.0044 | 0.7938 ± 0.0087 | 0.8000 ± 0.0049 | 3.9458 ± 2.8283
   0.009 | 0.7285 ± 0.0066 | 0.8079 ± 0.0046 | 0.7920 ± 0.0086 | 0.7998 ± 0.0054 | 3.9617 ± 2.8378
    0.01 | 0.7291 ± 0.0065 | 0.8090 ± 0.0041 | 0.7915 ± 0.0088 | 0.8001 ± 0.0055 | 3.9643 ± 2.8402
    0.05 | 0.7261 ± 0.0100 | 0.8202 ± 0.0054 | 0.7690 ± 0.0207 | 0.7935 ± 0.0103 | 3.9711 ± 2.7962
    0.09 | 0.7206 ± 0.0110 | 0.8259 ± 0.0066 | 0.7506 ± 0.0242 | 0.7862 ± 0.0120 | 4.0741 ± 2.7610
     0.1 | 0.7180 ± 0.0099 | 0.8268 ± 0.0086 | 0.7446 ± 0.0240 | 0.7832 ± 0.0112 | 4.1107 ± 2.7599
     0.3 | 0.7025 ± 0.0294 | 0.8325 ± 0.0194 | 0.7110 ± 0.0712 | 0.7640 ± 0.0393 | 4.4550 ± 2.7319
     0.5 | 0.7054 ± 0.0366 | 0.8298 ± 0.0251 | 0.7211 ± 0.0864 | 0.7673 ± 0.0481 | 4.5761 ± 2.8165
     0.9 | 0.7164 ± 0.0244 | 0.8220 ± 0.0269 | 0.7524 ± 0.0714 | 0.7825 ± 0.0322 | 4.7414 ± 2.9318
       1 | 0.7187 ± 0.0188 | 0.8204 ± 0.0270 | 0.7589 ± 0.0629 | 0.7858 ± 0.0254 | 4.7601 ± 2.9506
       2 | 0.7287 ± 0.0104 | 0.8163 ± 0.0257 | 0.7825 ± 0.0442 | 0.7976 ± 0.0136 | 4.8783 ± 2.9785
       5 | 0.7304 ± 0.0122 | 0.8121 ± 0.0286 | 0.7926 ± 0.0405 | 0.8008 ± 0.0110 | 4.9290 ± 2.9985
       8 | 0.7320 ± 0.0133 | 0.8133 ± 0.0306 | 0.7943 ± 0.0418 | 0.8021 ± 0.0110 | 4.9403 ± 3.0013
